In [3]:
%pip install --force-reinstall --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0
%pip install torch-geometric

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torch-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (28 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 118.1 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 310.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 134.7 MB/s  0:00:00
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 148.1 MB/s  0:00:02a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 151.7 MB/s  0:00:01a 0:00:01
     ━━━━

# SGPD (GSE145361) cohort

In [4]:
import os
import gc
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, GlobalAttention
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

/opt/conda/envs/rapids-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
FUNCTIONAL_MAP = {
    'TSS200': 0, 'TSS1500': 1, '1stExon': 2, 
    "5'UTR": 3, 'Body': 4, "3'UTR": 5, 'Other': 6
}

## Build the Chromosome topologies

In [6]:
def build_chromosome_topologies(manifest_df, common_probes, max_linear_dist_bp=1000):
    manifest = manifest_df[manifest_df['IlmnID'].isin(common_probes)].copy()
    manifest['CHR'] = manifest['CHR'].astype(str).str.replace('chr', '')
    manifest = manifest[manifest['CHR'].isin([str(i) for i in range(1, 23)])]
    manifest['MAPINFO'] = manifest['MAPINFO'].astype(int)
    
    chr_topologies = {}
    
    for c in range(1, 23):
        chr_str = str(c)
        sub = manifest[manifest['CHR'] == chr_str].sort_values(by='MAPINFO').reset_index(drop=True)
        probe_list = sub['IlmnID'].tolist()
        probe_to_idx = {p: i for i, p in enumerate(probe_list)}
        positions = sub['MAPINFO'].values
        n_nodes = len(probe_list)
        
        # A. Encode primary functional annotation
        func_labels = np.full(n_nodes, FUNCTIONAL_MAP['Other'], dtype=np.int64)
        for idx, row in sub.iterrows():
            grps = str(row.get('UCSC_RefGene_Group', '')).split(';')
            if grps and grps[0] in FUNCTIONAL_MAP:
                func_labels[idx] = FUNCTIONAL_MAP[grps[0]]
                
        # B. EXPLICIT SELF LOOPS (Fixes Empty Edge Crash & Mathematically sound for GAT)
        edges = [[i, i] for i in range(n_nodes)]
        
        # C. 1D Linear Adjacency Edges
        for i in range(n_nodes - 1):
            if 0 < (positions[i+1] - positions[i]) <= max_linear_dist_bp:
                edges.append([i, i + 1])
                edges.append([i + 1, i])
                
        # D. Shared Genic Annotation Edges
        gene_groups = {}
        for idx, row in sub.iterrows():
            genes = str(row.get('UCSC_RefGene_Name', '')).split(';')
            if genes and genes[0] not in ('', 'nan'):
                gene = genes[0]
                gene_groups.setdefault(gene, []).append(idx)
                
        for members in gene_groups.values():
            if 1 < len(members) <= 50:
                for i in range(len(members)):
                    for j in range(i + 1, len(members)):
                        edges.append([members[i], members[j]])
                        edges.append([members[j], members[i]])
                        
        # Because we initialized with self-loops, 'edges' is guaranteed non-empty
        edge_arr = np.unique(np.array(edges), axis=0).T
        edge_index = torch.tensor(edge_arr, dtype=torch.long)
            
        chr_topologies[c] = {
            'probes': probe_list,
            'edge_index': edge_index,
            'func_type': torch.tensor(func_labels, dtype=torch.long),
            'n_nodes': n_nodes
        }
        
    return chr_topologies

## PyTorch dataset and Trasposed Collator

In [7]:
LABEL_MAP = {'Control': 0.0, 'PD': 1.0}

class WholeBloodMethylationDataset(Dataset):
    def __init__(self, m_matrix_df, pheno_df, cell_cols, chr_topologies):
        self.sample_ids = pheno_df.index.tolist()
        
        # Explicit label encoding for BCEWithLogitsLoss
        labels = pheno_df['Sample_Group'].map(LABEL_MAP)
        if labels.isna().any():
            unknown_labels = sorted(pheno_df.loc[labels.isna(), 'Sample_Group'].astype(str).unique().tolist())
            raise ValueError(f"Unexpected Sample_Group values: {unknown_labels}")
        self.labels = labels.to_numpy(dtype=np.float32)
        
        self.cell_props = pheno_df[cell_cols].values.astype(np.float32)
        self.chr_topologies = chr_topologies
        
        self.chr_m_values = {}
        for c in range(1, 23):
            probes = chr_topologies[c]['probes']
            # Safe slice: guarantees no missing probes because df was pre-subsetted
            self.chr_m_values[c] = m_matrix_df.loc[self.sample_ids, probes].values.astype(np.float32)
            
    def __len__(self):
        return len(self.sample_ids)
        
    def __getitem__(self, idx):
        chr_graphs = []
        for c in range(1, 23):
            raw_x = torch.from_numpy(self.chr_m_values[c][idx]).unsqueeze(1)
            topo = self.chr_topologies[c]
            
            data = Data(
                x=raw_x,
                edge_index=topo['edge_index'],
                func_type=topo['func_type']
            )
            chr_graphs.append(data)
            
        u = torch.from_numpy(self.cell_props[idx])
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return chr_graphs, u, y


def chromosome_collate_fn(batch):
    batched_chromosomes = []
    graphs_per_sample = [item[0] for item in batch]
    u_tensor = torch.stack([item[1] for item in batch])
    y_tensor = torch.stack([item[2] for item in batch])
    
    for chr_idx in range(22):
        chr_list = [graphs[chr_idx] for graphs in graphs_per_sample]
        batched_chromosomes.append(Batch.from_data_list(chr_list))
        
    return batched_chromosomes, u_tensor, y_tensor

## Chromosome-Parallel GAT Architecture

In [8]:
class ChromosomeParallelGAT(nn.Module):
    def __init__(self, num_node_classes=7, chr_embed_dim=16, cell_prop_dim=6):
        super(ChromosomeParallelGAT, self).__init__()
        
        self.func_embedding = nn.Embedding(num_embeddings=num_node_classes, embedding_dim=8)
        
        # add_self_loops=False because we explicitly defined them in the topology builder
        self.gat1 = GATConv(in_channels=9, out_channels=8, heads=2, concat=True, add_self_loops=False)
        self.gat2 = GATConv(in_channels=16, out_channels=chr_embed_dim, heads=1, concat=True, add_self_loops=False)
        
        self.gate_nn = nn.Sequential(
            nn.Linear(chr_embed_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
        self.pool = GlobalAttention(gate_nn=self.gate_nn)
        
        fused_dim = (22 * chr_embed_dim) + cell_prop_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
        
    def forward(self, batched_chrs, u_cells):
        chr_embeddings = []
        for c_idx in range(22):
            batch = batched_chrs[c_idx]
            
            emb_func = self.func_embedding(batch.func_type)
            node_feat = torch.cat([batch.x, emb_func], dim=1)
            
            h = torch.relu(self.gat1(node_feat, batch.edge_index))
            h = torch.relu(self.gat2(h, batch.edge_index))
            
            chr_emb = self.pool(h, batch.batch)
            chr_embeddings.append(chr_emb)
            
        genome_vector = torch.cat(chr_embeddings, dim=1)
        fused = torch.cat([genome_vector, u_cells], dim=1)
        return self.classifier(fused)

## Data preps
___
Here we need to create distinct sets of pheno-data and CpG m-values. Those will be split off from the parquet file.

In [6]:
m_matrix_full = pd.read_parquet("/workspace/data/sgpd/GSE145361_data_corrected.parquet")
m_values_df = m_matrix_full.set_index("Sample_Name")

In [7]:
all_cols = m_values_df.columns.to_list()
probe_cols = [c for c in all_cols if c.startswith('cg') or (c.startswith('ch') and not c.startswith('cha'))]

In [8]:
probe_set = set(probe_cols)
pheno_cols = [c for c in all_cols if c not in probe_set]

In [10]:
pheno_df = m_values_df[pheno_cols].copy()

cat_cols = pheno_df.select_dtypes(include=["category"]).columns
for c in cat_cols:
    pheno_df[c] = pheno_df[c].astype("str")

pheno_out = "/workspace/data/sgpd/GSE145361_pheno_data.parquet"
pheno_df.to_parquet(pheno_out)
print(f"Wrote pheno_df to {pheno_out}")

Wrote pheno_df to /workspace/data/sgpd/GSE145361_pheno_data.parquet


In [11]:
manifest_path = "/workspace/data/infinium450k_manifest.parquet"
manifest_df = pd.read_parquet(manifest_path)

## 5-Fold CV

In [14]:

LABEL_MAP = {'Control': 0, 'PD': 1}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing GAT Pipeline on: {device}")

if pheno_df.index.name != "Sample_Name":
    pheno_df = pheno_df.set_index("Sample_Name")
    
cell_cols = ['CD8T', 'CD4T', 'NK', 'Bcell', 'Mono', 'Gran']
# Index is already set 
# m_values_df.set_index("Sample_Name", inplace=True)

# Strict validation to prevent index mismatch
common_probes = list(set(m_values_df.columns).intersection(set(manifest_df['IlmnID'])))
m_values_df = m_values_df[common_probes] # Discard unused columns to save RAM early
print(f"Total overlapping autosome probes: {len(common_probes):,}")

print("Building chromosome 1D adjacency and genic graphs...")
chr_topologies = build_chromosome_topologies(manifest_df, common_probes)

y = pheno_df['Sample_Group'].map(LABEL_MAP)
if y.isna().any():
    unknown_labels = sorted(pheno_df.loc[y.isna(), 'Sample_Group'].astype(str).unique().tolist())
    raise ValueError(f"Unexpected Sample_Group values: {unknown_labels}")
y = y.to_numpy(dtype=np.int64)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_aucs, pr_aucs = [], []
batch_size = 16  
epochs = 40
lr = 2e-4

for fold, (train_idx, test_idx) in enumerate(skf.split(pheno_df, y)):
    print(f"\n--- Fold {fold + 1} ---")
    train_pheno = pheno_df.iloc[train_idx]
    test_pheno = pheno_df.iloc[test_idx]
    
    train_ds = WholeBloodMethylationDataset(m_values_df, train_pheno, cell_cols, chr_topologies)
    test_ds = WholeBloodMethylationDataset(m_values_df, test_pheno, cell_cols, chr_topologies)
    
    train_loader = DataLoader(train_ds, 
                              batch_size=batch_size, 
                              shuffle=True, 
                              collate_fn=chromosome_collate_fn,
                              num_workers=16,
                              pin_memory=True,
                              persistent_workers=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=chromosome_collate_fn)
    
    model = ChromosomeParallelGAT().to(device)
    
    # Explicit dtype declaration for pos_weight
    train_labels = train_pheno['Sample_Group'].map(LABEL_MAP)
    if train_labels.isna().any():
        unknown_labels = sorted(train_pheno.loc[train_labels.isna(), 'Sample_Group'].astype(str).unique().tolist())
        raise ValueError(f"Unexpected Sample_Group values in training fold: {unknown_labels}")
    train_labels = train_labels.to_numpy(dtype=np.float32)
    positives = float((train_labels == 1.0).sum())
    negatives = float((train_labels == 0.0).sum())
    if positives == 0:
        raise ValueError("Training fold has no positive samples, cannot compute pos_weight")
    pos_weight = torch.tensor([negatives / positives], device=device, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    
    # Training
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for batched_chrs, u_cells, batch_y in train_loader:
            
            # Asynchronous memory transfer
            batched_chrs = [c.to(device, non_blocking=True) for c in batched_chrs]
            u_cells = u_cells.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            logits = model(batched_chrs, u_cells).squeeze(1)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # VRAM Fragmentation Protection
            del batched_chrs, u_cells, batch_y, logits, loss
            
    # Evaluation
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batched_chrs, u_cells, batch_y in test_loader:
            batched_chrs = [c.to(device, non_blocking=True) for c in batched_chrs]
            u_cells = u_cells.to(device, non_blocking=True)
            
            logits = model(batched_chrs, u_cells).squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            preds.extend(probs)
            truths.extend(batch_y.numpy())
            
            del batched_chrs, u_cells, logits
            
    fold_roc = roc_auc_score(truths, preds)
    fold_pr = average_precision_score(truths, preds)
    roc_aucs.append(fold_roc)
    pr_aucs.append(fold_pr)
    print(f"Fold {fold + 1} | ROC AUC: {fold_roc:.4f} | PR AUC: {fold_pr:.4f}")

    # Save the model weights for downstream biological extraction
    torch.save(model.state_dict(), f"/workspace/results/sgpd/gat_fold_{fold + 1}.pt")

    del model, optimizer, train_ds, test_ds
    torch.cuda.empty_cache()
    gc.collect()
    
print("\n" + "=" * 40)
print(f"Mean GAT ROC AUC: {np.mean(roc_aucs):.4f} ± {np.std(roc_aucs):.4f}")
print(f"Mean GAT PR AUC:  {np.mean(pr_aucs):.4f} ± {np.std(pr_aucs):.4f}")

Executing GAT Pipeline on: cuda
Total overlapping autosome probes: 421,350
Building chromosome 1D adjacency and genic graphs...

--- Fold 1 ---


/tmp/ipykernel_2966/2041644710.py:16: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)


Fold 1 | ROC AUC: 0.6742 | PR AUC: 0.7026

--- Fold 2 ---


/tmp/ipykernel_2966/2041644710.py:16: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)


Fold 2 | ROC AUC: 0.6589 | PR AUC: 0.6871

--- Fold 3 ---


...

Fold 3 | ROC AUC: 0.6875 | PR AUC: 0.6731

--- Fold 4 ---


/tmp/ipykernel_2966/2041644710.py:16: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)


Fold 4 | ROC AUC: 0.6622 | PR AUC: 0.6561

--- Fold 5 ---


/tmp/ipykernel_2966/2041644710.py:16: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)


Fold 5 | ROC AUC: 0.6320 | PR AUC: 0.6250

Mean GAT ROC AUC: 0.6630 ± 0.0185
Mean GAT PR AUC:  0.6688 ± 0.0268


In [9]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch_geometric.nn import GATConv, GlobalAttention
from torch_geometric.utils import softmax
from torch_geometric.data import Batch, Data

# =====================================================================
# 1. THE EXPLAINER ARCHITECTURE
# =====================================================================
# This is identical to your training architecture, but the forward pass 
# is modified to explicitly return the biological attention weights.
class GATExplainer(nn.Module):
    def __init__(self, num_node_classes=7, chr_embed_dim=16, cell_prop_dim=6):
        super(GATExplainer, self).__init__()
        self.func_embedding = nn.Embedding(num_embeddings=num_node_classes, embedding_dim=8)
        self.gat1 = GATConv(in_channels=9, out_channels=8, heads=2, concat=True, add_self_loops=False)
        self.gat2 = GATConv(in_channels=16, out_channels=chr_embed_dim, heads=1, concat=True, add_self_loops=False)
        
        self.gate_nn = nn.Sequential(
            nn.Linear(chr_embed_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
        self.pool = GlobalAttention(gate_nn=self.gate_nn)
        
        fused_dim = (22 * chr_embed_dim) + cell_prop_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, batched_chrs, u_cells):
        genome_attention = {}
        
        for c_idx in range(22):
            batch = batched_chrs[c_idx]
            
            # Forward pass through GAT layers
            emb_func = self.func_embedding(batch.func_type)
            node_feat = torch.cat([batch.x, emb_func], dim=1)
            h = torch.relu(self.gat1(node_feat, batch.edge_index))
            h = torch.relu(self.gat2(h, batch.edge_index))
            
            # --- EXTRACT BIOLOGICAL IMPORTANCE ---
            # Calculate the raw importance score for every single CpG site
            raw_gate_scores = self.gate_nn(h).view(-1)
            # Normalize scores so they sum to 1.0 across the chromosome
            cpg_attention = softmax(raw_gate_scores, batch.batch)
            
            # Store the attention weights for this chromosome
            # Shape: [num_nodes_in_batch]
            genome_attention[c_idx + 1] = cpg_attention.detach().cpu().numpy()
            
        return genome_attention

# =====================================================================
# 2. EXTRACTION LOGIC
# =====================================================================
def extract_biological_drivers(model_path, m_matrix_df, pheno_df, chr_topologies, cell_cols):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Extracting on: {device}")
    
    # 1. Initialize and Load Weights
    model = GATExplainer().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # 2. Isolate Parkinson's Patients (We want to see what drives the PD signature)
    pd_samples = pheno_df[pheno_df['Sample_Group'] == 'PD'].index.tolist()
    print(f"Extracting drivers across {len(pd_samples)} PD patients...")
    
    # 3. Process Patients (Batch size 1 for clean extraction)
    # We will accumulate the attention weights to find the consensus drivers
    consensus_attention = {c: np.zeros(chr_topologies[c]['n_nodes']) for c in range(1, 23)}
    
    for sample in pd_samples:
        chr_graphs = []
        for c in range(1, 23):
            probes = chr_topologies[c]['probes']
            raw_x = torch.from_numpy(m_matrix_df.loc[sample, probes].values.astype(np.float32)).unsqueeze(1)
            data = Data(x=raw_x, edge_index=chr_topologies[c]['edge_index'], func_type=chr_topologies[c]['func_type'])
            chr_graphs.append(data)
            
        # Convert to PyG Batch format (batch size 1)
        batched_chrs = [Batch.from_data_list([g]).to(device) for g in chr_graphs]
        u = torch.tensor(pheno_df.loc[sample, cell_cols].values.astype(np.float32)).unsqueeze(0).to(device)
        
        # 4. Extract Attention
        with torch.no_grad():
            attention_dict = model(batched_chrs, u)
            
        for c in range(1, 23):
            consensus_attention[c] += attention_dict[c]

    # 5. Average the attention across all PD patients
    for c in range(1, 23):
        consensus_attention[c] /= len(pd_samples)
        
    return consensus_attention

# =====================================================================
# 3. MAP TO BIOLOGY & EXPORT
# =====================================================================
def map_and_export_drivers(consensus_attention, chr_topologies, manifest_df, export_path="pd_epigenetic_drivers.csv"):
    records = []
    
    for c in range(1, 23):
        probes = chr_topologies[c]['probes']
        weights = consensus_attention[c]
        
        for idx, probe in enumerate(probes):
            records.append({
                'IlmnID': probe,
                'Chromosome': c,
                'Attention_Score': weights[idx]
            })
            
    df_weights = pd.DataFrame(records)
    
    # Merge with Manifest to get Genes and Regions
    final_df = df_weights.merge(
        manifest_df[['IlmnID', 'UCSC_RefGene_Name', 'UCSC_RefGene_Group']], 
        on='IlmnID', 
        how='left'
    )
    
    # Sort by highest attention score
    final_df = final_df.sort_values(by='Attention_Score', ascending=False).reset_index(drop=True)
    final_df.to_csv(export_path, index=False)
    print(f"Top drivers exported to {export_path}")
    
    return final_df



In [10]:
manifest_path = "/workspace/data/infinium450k_manifest.parquet"
manifest_df = pd.read_parquet(manifest_path)

m_matrix_full = pd.read_parquet("/workspace/data/sgpd/GSE145361_data_corrected.parquet")
m_values_df = m_matrix_full.set_index("Sample_Name")

common_probes = list(set(m_values_df.columns).intersection(set(manifest_df['IlmnID'])))

chr_topologies = build_chromosome_topologies(manifest_df, common_probes)

pheno_df = pd.read_parquet("/workspace/results/sgpd/GSE145361_pheno_data.parquet")

In [11]:
torch.cuda.is_available()

True

In [12]:
cell_cols = ['CD8T', 'CD4T', 'NK', 'Bcell', 'Mono', 'Gran']

# 1. Define the paths to your 5 saved models
fold_paths = [f"/workspace/results/sgpd/gat_fold_{i}.pt" for i in range(1, 6)]

# 2. Initialize a master dictionary to hold the ensemble weights
master_consensus = {c: np.zeros(chr_topologies[c]['n_nodes']) for c in range(1, 23)}

print("Starting Ensemble Extraction across 5 folds...")

# 3. Loop through each fold, extract weights, and add to the master dictionary
for fold_idx, model_path in enumerate(fold_paths):
    print(f"\n--- Processing {model_path} ---")
    
    # Run the extraction function we defined earlier
    fold_consensus = extract_biological_drivers(
        model_path, 
        m_values_df, 
        pheno_df, 
        chr_topologies, 
        cell_cols
    )
    
    # Add this fold's weights to the master tracker
    for c in range(1, 23):
        master_consensus[c] += fold_consensus[c]

# 4. Average the weights across the 5 folds
for c in range(1, 23):
    master_consensus[c] /= 5.0

print("\nEnsemble extraction complete! Mapping to Illumina Manifest...")

# 5. Map the ensemble weights to genes and export
ensemble_drivers_df = map_and_export_drivers(
    master_consensus, 
    chr_topologies, 
    manifest_df, 
    export_path="/workspace/results/pd_ensemble_epigenetic_drivers.csv"
)

# Display the top 20 most critical biological drivers
print("\nTop 20 Consistently Weighted CpG Sites:")
print(ensemble_drivers_df.head(20))

Starting Ensemble Extraction across 5 folds...

--- Processing /workspace/results/sgpd/gat_fold_1.pt ---
Extracting on: cuda


/tmp/ipykernel_2981/3677629239.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)


Extracting drivers across 927 PD patients...

--- Processing /workspace/results/sgpd/gat_fold_2.pt ---
Extracting on: cuda
Extracting drivers across 927 PD patients...


/tmp/ipykernel_2981/3677629239.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)



--- Processing /workspace/results/sgpd/gat_fold_3.pt ---
Extracting on: cuda
Extracting drivers across 927 PD patients...


...


--- Processing /workspace/results/sgpd/gat_fold_4.pt ---
Extracting on: cuda
Extracting drivers across 927 PD patients...


/tmp/ipykernel_2981/3677629239.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)



--- Processing /workspace/results/sgpd/gat_fold_5.pt ---
Extracting on: cuda
Extracting drivers across 927 PD patients...


/tmp/ipykernel_2981/3677629239.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)



Ensemble extraction complete! Mapping to Illumina Manifest...
Top drivers exported to /workspace/results/pd_ensemble_epigenetic_drivers.csv

Top 20 Consistently Weighted CpG Sites:
        IlmnID  Chromosome  Attention_Score             UCSC_RefGene_Name  \
0   cg07201638          21         0.000300                        CLDN17   
1   cg02825728          21         0.000300                        CLDN17   
2   cg13792279          21         0.000300                        CLDN17   
3   cg24936695          21         0.000300                        CLDN17   
4   cg24343720          21         0.000300                        CLDN17   
5   cg00707688          21         0.000299  KRTAP10-5;C21orf29;KRTAP10-5   
6   cg04052038          21         0.000298                         CLDN8   
7   cg19026320          21         0.000298                         CLDN8   
8   cg22540233          21         0.000298                         CLDN8   
9   cg06427867          21         0.000298     

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from torch.utils.data import DataLoader

def generate_evaluation_plots(m_matrix_df, pheno_df, chr_topologies, cell_cols):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Recreate the exact splits
    y = pheno_df['Sample_Group'].map({'Control': 0, 'PD': 1}).to_numpy(dtype=np.int64)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Trackers for mean curves
    tprs, aucs = [], []
    mean_fpr = np.linspace(0, 1, 100)
    
    pr_aucs = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(pheno_df, y)):
        print(f"Evaluating Fold {fold + 1}...")
        test_pheno = pheno_df.iloc[test_idx]
        test_ds = WholeBloodMethylationDataset(m_matrix_df, test_pheno, cell_cols, chr_topologies)
        test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=chromosome_collate_fn)
        
        # 2. Load the saved weights
        model = ChromosomeParallelGAT().to(device)
        model.load_state_dict(torch.load(f"/workspace/results/gat_fold_{fold + 1}.pt", map_location=device))
        model.eval()
        
        # 3. Get Predictions
        preds, truths = [], []
        with torch.no_grad():
            for batched_chrs, u_cells, batch_y in test_loader:
                batched_chrs = [c.to(device) for c in batched_chrs]
                logits = model(batched_chrs, u_cells.to(device)).squeeze(1)
                preds.extend(torch.sigmoid(logits).cpu().numpy())
                truths.extend(batch_y.numpy())
                
        # 4. Calculate ROC Metrics
        fpr, tpr, _ = roc_curve(truths, preds)
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        ax_roc.plot(fpr, tpr, alpha=0.3, label=f'Fold {fold+1} (AUC = {roc_auc:.2f})')
        
        # 5. Calculate PR Metrics
        precision, recall, _ = precision_recall_curve(truths, preds)
        pr_auc = average_precision_score(truths, preds)
        pr_aucs.append(pr_auc)
        ax_pr.plot(recall, precision, alpha=0.3, label=f'Fold {fold+1} (AP = {pr_auc:.2f})')

    # --- Plot Formatting ---
    # ROC Mean
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    std_auc = np.std(aucs)
    ax_roc.plot(mean_fpr, mean_tpr, color='b', label=f'Mean ROC (AUC = {mean_auc:.2f} $\pm$ {std_auc:.2f})', lw=2)
    ax_roc.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', label='Chance')
    ax_roc.set(xlim=[-0.05, 1.05], ylim=[-0.05, 1.05], title="Receiver Operating Characteristic (ROC)", xlabel="False Positive Rate", ylabel="True Positive Rate")
    ax_roc.legend(loc="lower right")
    
    # PR Mean
    mean_pr = np.mean(pr_aucs)
    std_pr = np.std(pr_aucs)
    baseline = np.sum(y) / len(y)
    ax_pr.axhline(y=baseline, color='r', linestyle='--', label=f'Baseline ({baseline:.2f})')
    ax_pr.set(xlim=[-0.05, 1.05], ylim=[-0.05, 1.05], title=f"Precision-Recall Curve (Mean AP = {mean_pr:.2f} $\pm$ {std_pr:.2f})", xlabel="Recall", ylabel="Precision")
    ax_pr.legend(loc="lower left")
    
    plt.tight_layout()
    plt.savefig("/workspace/results/gat_cv_metrics.pdf", format="pdf", bbox_inches="tight")
    plt.show()